# T03 — Load and recode PISA 2022 data

This notebook runs the readable preparation script and inspects its outputs. The six systems are UAE (`ARE`), Saudi Arabia (`SAU`), Qatar (`QAT`), Jordan (`JOR`), Morocco (`MAR`) and Palestinian Authority (`PSE`).

The script selects 17 predictors, retains all ten mathematics plausible values and the final student weight plus 80 replicate weights, and joins school information using `CNT` and `CNTSCHID`. A left join preserves every student; `validate="many_to_one"` prevents duplicate school records from multiplying student rows.

Missing codes are identified from each variable's dictionary labels. Codes differ between variables, so a global replacement would be unsafe. The existing Parquet data already store these missing answers as `NaN`; rerunning the recoding is safe. Each predictor gets an indicator: **1 = missing, 0 = present**.

T03 leaves imputation, scaling, categorical encoding, target construction and modelling to later tasks.

In [1]:
from pathlib import Path
import runpy
import pandas as pd

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / "data/data_dictionary.csv").exists())
script = ROOT / "02_data-analysis/scripts/t03_prepare_data.py"
preparation = runpy.run_path(str(script))
data, audit = preparation["main"]()
OUTPUT = ROOT / "02_data-analysis/outputs/t03"

     students  schools  missing_or_nonpositive_weights
CNT                                                   
ARE     24600      840                               0
SAU      6928      193                               0
QAT      7676      229                               0
JOR      7799      260                               0
MAR      6867      178                               0
PSE      7905      273                               0

Saved 61,775 students and 128 columns to /Users/annierajesh/Documents/F21DL-PISA-2022/02_data-analysis/outputs/t03
Unmatched students: 0; newly recoded values: 0


## Check the join
The before/after counts must agree. Duplicate students and unmatched students should be zero for this data.

In [2]:
audit

{'systems': ['ARE', 'SAU', 'QAT', 'JOR', 'MAR', 'PSE'],
 'students_before_join': 61775,
 'students_after_join': 61775,
 'school_records_loaded': 1973,
 'schools_with_students': 1973,
 'duplicate_students': 0,
 'unmatched_students': 0,
 'unmatched_schools': 0,
 'predictors': ['ESCS',
  'PAREDINT',
  'HISEI',
  'HOMEPOS',
  'ANXMAT',
  'MATHEFF',
  'MATHEF21',
  'BELONG',
  'BULLIED',
  'DISCLIM',
  'TEACHSUP',
  'EDUSHORT',
  'STAFFSHORT',
  'MCLSIZE',
  'STRATIO',
  'SCHLTYPE',
  'SCHSIZE'],
 'missing_indicators': 17,
 'mathematics_plausible_values': ['PV1MATH',
  'PV2MATH',
  'PV3MATH',
  'PV4MATH',
  'PV5MATH',
  'PV6MATH',
  'PV7MATH',
  'PV8MATH',
  'PV9MATH',
  'PV10MATH'],
 'weight_columns': 81,
 'newly_recoded_values': 0,
 'missing_mathematics_values': 0,
 'missing_or_nonpositive_weight_values': 0}

## Count students and schools
These are unweighted sample counts, not population estimates.

In [3]:
pd.read_csv(OUTPUT / "system_summary.csv")

,CNT,students,schools,missing_or_nonpositive_weights
0,ARE,24600,840,0
1,SAU,6928,193,0
2,QAT,7676,229,0
3,JOR,7799,260,0
4,MAR,6867,178,0
5,PSE,7905,273,0


## Inspect missingness
`already_missing` counts existing NaNs; `newly_recoded` counts additional documented codes replaced in this run. A matched school can still have missing questionnaire answers.

In [4]:
pd.read_csv(OUTPUT / "missingness.csv")

,variable,source,documented_missing_codes,already_missing,newly_recoded,total_missing,percent_missing
0,ESCS,student,"[95.0, 97.0, 98.0, 99.0]",2432,0,2432,3.936868
1,PAREDINT,student,"[9995.0, 9997.0, 9998.0, 9999.0]",2659,0,2659,4.304330
2,HISEI,student,"[995.0, 997.0, 998.0, 999.0]",9017,0,9017,14.596520
3,HOMEPOS,student,"[95.0, 97.0, 98.0, 99.0]",1764,0,1764,2.855524
4,ANXMAT,student,"[95.0, 97.0, 98.0, 99.0]",14123,0,14123,22.861999
5,MATHEFF,student,"[95.0, 97.0, 98.0, 99.0]",13037,0,13037,21.104006
6,MATHEF21,student,"[95.0, 97.0, 98.0, 99.0]",13837,0,13837,22.399029
7,BELONG,student,"[95.0, 97.0, 98.0, 99.0]",5714,0,5714,9.249696
8,BULLIED,student,"[95.0, 97.0, 98.0, 99.0]",5572,0,5572,9.019830
9,DISCLIM,student,"[95.0, 97.0, 98.0, 99.0]",10455,0,10455,16.924322


In [5]:
pd.read_csv(OUTPUT / "missingness_by_system.csv").round(2)

,CNT,ESCS,PAREDINT,HISEI,HOMEPOS,ANXMAT,MATHEFF,MATHEF21,BELONG,BULLIED,DISCLIM,TEACHSUP,EDUSHORT,STAFFSHORT,MCLSIZE,STRATIO,SCHLTYPE,SCHSIZE
0,ARE,5.66,5.84,15.29,4.48,16.91,15.82,16.63,9.49,9.23,13.62,14.28,18.93,18.91,19.07,21.21,10.01,15.29
1,JOR,2.08,2.26,13.50,0.95,28.27,25.84,27.57,10.72,10.03,20.13,21.84,0.00,0.00,0.00,0.76,0.00,0.76
2,MAR,0.45,1.82,5.45,0.22,37.44,33.87,35.53,5.69,5.75,23.82,26.07,4.11,4.11,4.11,34.22,0.00,33.17
3,PSE,2.59,2.64,14.61,1.58,22.05,19.91,21.76,9.29,9.06,15.91,17.03,0.00,0.00,0.00,1.29,0.00,0.86
4,QAT,6.51,7.32,18.20,4.59,30.39,28.52,30.18,12.96,12.82,23.32,24.82,0.00,0.00,0.00,0.27,0.00,0.27
5,SAU,2.05,2.17,18.42,1.40,16.05,15.04,16.17,6.11,6.12,12.27,12.73,0.84,0.84,0.26,11.65,0.00,7.49


## Inspect the prepared rows
The saved Parquet is the input for subsequent tasks. Missingness indicators describe predictors only; scores and weights are retained separately.

In [6]:
data.head()

,CNT,CNTSCHID,CNTSTUID,ESCS,PAREDINT,HISEI,HOMEPOS,ANXMAT,MATHEFF,MATHEF21,...,BELONG_missing,BULLIED_missing,DISCLIM_missing,TEACHSUP_missing,EDUSHORT_missing,STAFFSHORT_missing,MCLSIZE_missing,STRATIO_missing,SCHLTYPE_missing,SCHSIZE_missing
0,PSE,27500099.0,27500001.0,-0.6594,16.0,NaN,-0.5451,NaN,-1.7181,NaN,...,0,0,1,1,0,0,0,0,0,0
1,PSE,27500050.0,27500002.0,-0.7512,12.0,75.43,-1.8665,0.4067,-1.2825,-0.5623,...,0,0,0,0,0,0,0,0,0,0
2,PSE,27500157.0,27500003.0,-0.7349,16.0,16.50,-0.7994,0.9050,-0.3051,-0.2354,...,0,0,0,0,0,0,0,0,0,0
3,PSE,27500213.0,27500004.0,1.6743,16.0,76.65,2.2074,NaN,0.0501,NaN,...,0,0,1,1,0,0,0,0,0,0
4,PSE,27500038.0,27500005.0,-1.7778,12.0,23.57,-1.9787,0.4290,0.1033,-2.0992,...,0,0,0,0,0,0,0,0,0,0
